In [1]:
import pandas as pd
import numpy as np
import math
import re
from collections import Counter, defaultdict
import pickle
import json

In [2]:
# defining the path to the dataset
dataset_path = r'D:/JantaKoAwaj-FYP/jka-ml-model/dataset/preprocessed_data.pkl'

# reading the dataset saved in pickle format
df = pd.read_pickle(dataset_path)
print(df.head())


                  Brief Description of the grievance  \
0  Mero ghar ko aagan ma dhulo dherai aauchha. Dh...   
1  Mero galli ma hidda mero chhaya dherai lamo bh...   
2  वडा नं. १६ नवोदित स्कुल नजिकै द्याबु मार्गमा व...   
3  नदीमा माछाहरू धेरै छन्। यिनीहरूले पानी फोहोर ग...   
4   ३१ वडा पञ्चकुमारी मन्दिर निरकाे खालि जग्गामा ...   

                                        cleaned_text  \
0  mero ghar ko aagan ma dhulo dherai aauchha. dh...   
1  mero galli ma hidda mero chhaya dherai lamo bh...   
2  वडा नं. १६ नवोदित स्कुल नजिकै द्याबु मार्गमा व...   
3  नदीमा माछाहरू धेरै छन्। यिनीहरूले पानी फोहोर ग...   
4  ३१ वडा पञ्चकुमारी मन्दिर निरकाे खालि जग्गामा ल...   

                             stopword_removed_tokens        Label  
0  [ghar, aagan, dhulo, aauchha, ., dhulo, rokna,...  not_genuine  
1  [galli, hidda, chhaya, lamo, bhayo, ., chhaya,...  not_genuine  
2  [वडा, नं, ., १६, नवोदित, स्कुल, द्याबु, मार्गम...      genuine  
3  [नदीमा, माछाहरू, छन्।, यिनीहरूले, पानी, फोहोर,...  

In [3]:
class TFIDFVectorizer:
    def __init__(self, max_features=10000, ngram_range=(1, 2), min_df=1, max_df=0.95):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.max_df = max_df
        self.vocabulary_ = {}
        self.idf_ = {}
        self.feature_names_ = []
        self.n_docs_ = 0
        
    def _tokenize(self, text):
        tokens = re.findall(r'\b\w+\b', text.lower())
        return tokens

    def _generate_ngrams(self, tokens, n):
        if n == 1:
            return tokens
        return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    def _get_ngrams_from_text(self, text):
        tokens = self._tokenize(text)
        all_ngrams = []
        for n in range(self.ngram_range[0], self.ngram_range[1]+1):
            all_ngrams.extend(self._generate_ngrams(tokens, n))
        return all_ngrams
    
    def _build_vocabulary(self, documents):
        doc_freq = defaultdict(int)
        total_docs = len(documents)
        
        for doc in documents:
            terms = set(self._get_ngrams_from_text(doc))
            for term in terms:
                doc_freq[term] += 1
        
        filtered_terms = {term: freq for term, freq in doc_freq.items() 
                          if freq >= self.min_df and (freq/total_docs) <= self.max_df}
        
        sorted_terms = sorted(filtered_terms.items(), key=lambda x: x[1], reverse=True)
        if self.max_features:
            sorted_terms = sorted_terms[:self.max_features]
        
        self.vocabulary_ = {term: idx for idx, (term, _) in enumerate(sorted_terms)}
        self.feature_names_ = [term for term, _ in sorted_terms]
        self.n_docs_ = total_docs
        
        for term, idx in self.vocabulary_.items():
            df = filtered_terms[term]
            self.idf_[idx] = math.log(total_docs / df)
    
    def _calculate_tf(self, document):
        terms = self._get_ngrams_from_text(document)
        term_count = Counter(terms)
        total_terms = len(terms)
        tf_vector = np.zeros(len(self.vocabulary_))
        
        for term, count in term_count.items():
            if term in self.vocabulary_:
                idx = self.vocabulary_[term]
                tf_vector[idx] = count / total_terms if total_terms > 0 else 0
        return tf_vector
    
    def fit(self, documents):
        self._build_vocabulary(documents)
        return self
    
    def transform(self, documents):
        if not self.vocabulary_:
            raise ValueError("Vectorizer has not been fitted yet!")
        tfidf_matrix = []
        for doc in documents:
            tf_vector = self._calculate_tf(doc)
            tfidf_vector = np.array([tf_vector[idx] * self.idf_[idx] for idx in range(len(tf_vector))])
            tfidf_matrix.append(tfidf_vector)
        return np.array(tfidf_matrix)
    
    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)
    
    def get_feature_names(self):
        return self.feature_names_
    
    def save_vectorizer(self, filepath):
        vectorizer_data = {
            'vocabulary_': self.vocabulary_,
            'idf_': self.idf_,
            'feature_names_': self.feature_names_,
            'n_docs_': self.n_docs_,
            'max_features': self.max_features,
            'ngram_range': self.ngram_range,
            'min_df': self.min_df,
            'max_df': self.max_df
        }
        with open(filepath, 'wb') as f:
            pickle.dump(vectorizer_data, f)

    @classmethod
    def load_vectorizer(cls, filepath):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        vectorizer = cls(max_features=data['max_features'],
                         ngram_range=data['ngram_range'],
                         min_df=data['min_df'],
                         max_df=data['max_df'])
        vectorizer.vocabulary_ = data['vocabulary_']
        vectorizer.idf_ = data['idf_']
        vectorizer.feature_names_ = data['feature_names_']
        vectorizer.n_docs_ = data['n_docs_']
        return vectorizer

In [4]:
vectorizer = TFIDFVectorizer(max_features=10000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df['cleaned_text'])

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Number of features:", len(vectorizer.get_feature_names()))


TF-IDF matrix shape: (9874, 10000)
Number of features: 10000


In [5]:
# Save TF-IDF matrix
np.save(r'D:/JantaKoAwaj-FYP/jka-ml-model/dataset/features/custom_tfidf_features.npy', X_tfidf)

# Save vectorizer
vectorizer.save_vectorizer(r'D:/JantaKoAwaj-FYP/jka-ml-model/dataset/features/custom_tfidf_vectorizer.pkl')

# Save labels
df['labeled'] = df['Label'].map({'not_genuine': 0, 'genuine': 1})
df['labeled'].to_csv(r'D:/JantaKoAwaj-FYP/jka-ml-model/dataset/features/custom_labeled_data.csv', index=False)

print("Saved TF-IDF features, vectorizer, and labels successfully!")


Saved TF-IDF features, vectorizer, and labels successfully!
